# Lab 2: Transformers and attention

This lab is about Transformers, the architecture behind practically all modern language models, and many models for other modalities.

The lab has three parts:

1. **Theory intro** - look inside pretrained Transformers with the HuggingFace `transformers` library

2. **High-level usage** - applications of pretrained Transformers across various linguistic tasks

3. **Architecture deep-dive** - implementing elements of modern Transformers

Exercises to do are marked. In most places, you are supposed to fill in the `...` gaps. Subsequent cells have checks for your solution.

**Hardware** - all models are small (at most 2B parameters) and run on a laptop. A GPU is faster, and free Google Colab T4 is enough (*Runtime -> Change runtime type -> T4 GPU*). On a MacBook, the Apple GPU (MPS) is used automatically. CPU works too, but slower.

# Environment setup

You can use a local environment with the [uv](https://docs.astral.sh/uv/) package manager, or [Google Colab](https://colab.research.google.com/). Everything was tested with `uv`: if you install packages manually with `pip`, something may break because of incompatible versions.

### Local environment

1. Install uv [following the documentation](https://docs.astral.sh/uv/getting-started/installation/), if you don't have it yet
2. In the directory of this lab, install the dependencies from `uv.lock`: `uv sync`
3. Run Jupyter Notebook: `uv run jupyter notebook`

### Google Colab

Upload `requirements.txt` from the directory of this lab to `/content`, and run `!pip install -r requirements.txt` in the cell below. If Colab asks you to restart the session afterwards, do it and continue with the next cell.

### Models and datasets

Models and datasets are downloaded from the [HuggingFace Hub](https://huggingface.co/) on first use and cached in `~/.cache/huggingface`, about 7 GB in total. None of them requires an account.

In [ ]:
# uncomment and run if using Google Colab
# !pip install -r requirements.txt

In [ ]:
import gc
import json
import math
import os
import time
import warnings
from collections.abc import Callable, Iterator
from dataclasses import dataclass
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import disable_progress_bars, load_dataset
from matplotlib_inline.backend_inline import set_matplotlib_formats
from sentence_transformers import SentenceTransformer
from tokenizers import Tokenizer, models, trainers
from torch.nn.attention import SDPBackend, sdpa_kernel
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    pipeline,
)
from transformers.models.llama.modeling_llama import (
    LlamaRotaryEmbedding,
    apply_rotary_pos_emb,
)

# variable-length FlashAttention, used in Part 3
try:
    from torch.nn.attention.varlen import varlen_attn
except ImportError:
    varlen_attn = None

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# 16-bit precision for the larger models: bfloat16 on modern NVIDIA GPUs,
# float16 on older GPUs (e.g. Colab T4) and Apple GPUs, float32 on CPU
if device.type == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False):
    HALF_DTYPE = torch.bfloat16
elif device.type in ("cuda", "mps"):
    HALF_DTYPE = torch.float16
else:
    HALF_DTYPE = torch.float32

print(
    f"PyTorch {torch.__version__}, transformers {transformers.__version__}, "
    f"device: {device}, 16-bit dtype: {HALF_DTYPE}"
)

In [ ]:
# computed embeddings are saved here, so that they persist after a restart
DATA_ROOT = "data"
os.makedirs(DATA_ROOT, exist_ok=True)

# models used in this lab, all from the HuggingFace Hub
ST_MODEL = "MongoDB/mdbr-leaf-mt"  # Sentence Transformer
LLM_EMBEDDING_MODEL = "microsoft/harrier-oss-v1-270m"  # LLM-based embeddings
LLM_MODEL = "Qwen/Qwen3-1.7B"  # text generation
ENCODER_MODEL = "jhu-clsp/ettin-encoder-68m"  # BERT-style encoder
DECODER_MODEL = "jhu-clsp/ettin-decoder-68m"  # GPT-style decoder


def free_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()


transformers.set_seed(0)
plt.rcParams.update(
    {"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False}
)
# JPEG instead of PNG figures to make notebook smaller
set_matplotlib_formats("jpeg", pil_kwargs={"quality": 90})

# hide progress bars and informational messages
transformers.logging.disable_progress_bar()
transformers.logging.set_verbosity_error()
disable_progress_bars()

# Part 1: Theory intro

The [HuggingFace `transformers`](https://huggingface.co/docs/transformers/index) library gives a uniform interface to hundreds of thousands of pretrained models on the HuggingFace Hub:

- `AutoTokenizer` turns text into token ids and back
- `AutoModel` turns token ids into token embeddings (hidden states), with variants like `AutoModelForCausalLM` adding a task-specific head
- `pipeline` wraps both for common tasks in one line

### 1.1 Tokenization

The **tokenizer** splits text into **subwords**: frequent words stay whole, rare words are split into pieces, so there are no unknown words. Every token id selects one row of a learned **embedding table** of size `vocabulary x d`. Let's compare the tokenizers of the models used in this lab:

* `mdbr-leaf-mt` - a small BERT, WordPiece with 30k tokens (`##` continues a word)
* Ettin - a modern encoder from the ModernBERT family, byte-level BPE with 50k tokens (`Ġ` = preceding space)
* Qwen3 - an LLM, byte-level BPE with 151k tokens
* Harrier - an embedding model built from the Gemma 3 LLM, BPE with 262k tokens (`▁` = space)

Note the **special tokens** (`[CLS]`, `[SEP]`, `<bos>`, `<eos>`), and how numbers are split.

In [ ]:
tokenizers = {
    "BERT (leaf-mt)": AutoTokenizer.from_pretrained(ST_MODEL),
    "ModernBERT (Ettin)": AutoTokenizer.from_pretrained(ENCODER_MODEL),
    "Qwen3": AutoTokenizer.from_pretrained(LLM_MODEL),
    "Gemma 3 (Harrier)": AutoTokenizer.from_pretrained(LLM_EMBEDDING_MODEL),
}

text = "The unbelievably fast tokenizer splits 'transformerization' into 1234 pieces."
for name, tokenizer in tokenizers.items():
    input_ids = tokenizer(text)["input_ids"]
    print(f"{name}, vocabulary {len(tokenizer):,}, {len(input_ids)} tokens:")
    print(f"  ids:    {input_ids}")
    print(f"  tokens: {tokenizer.convert_ids_to_tokens(input_ids)}\n")

A larger vocabulary gives shorter sequences (cheaper attention), but a larger embedding table. For small models with huge vocabularies, the table can be most of the model!

In [ ]:
texts = {
    "English": "Attention lets every token gather information from all other tokens, "
    "with weights computed from the data.",
    "Python": "def softmax(x):\n    e = np.exp(x - x.max())\n    return e / e.sum()",
    "numbers": "3.14159 2.71828 1.41421 299792458 6.02214076e23",
    "RNA": "AUGGCUACGUAGCUAGCUAGGCUAUCGAUCG",
}
token_counts = {
    name: {kind: len(tokenizer.tokenize(t)) for kind, t in texts.items()}
    for name, tokenizer in tokenizers.items()
}
print("number of tokens:")
print(pd.DataFrame(token_counts).T.to_string(), "\n")

# configs only, without downloading weights
for name, model_name, total_params in [
    ("BERT (leaf-mt)", ST_MODEL, 23e6),
    ("Gemma 3 (Harrier)", LLM_EMBEDDING_MODEL, 268e6),
    ("Qwen3", LLM_MODEL, 1.72e9),
]:
    config = transformers.AutoConfig.from_pretrained(model_name)
    embedding_params = config.vocab_size * config.hidden_size
    print(
        f"{name}: embedding table {config.vocab_size:,} x {config.hidden_size} = "
        f"{embedding_params / 1e6:.0f}M params, {100 * embedding_params / total_params:.0f}% of the model"
    )

### 1.2 Scaled dot-product attention (SDPA)

Each token emits a query $q$, a key $k$ and a value $v$. The output for a token is the average of all values, weighted by the softmax of query-key similarities:

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

In PyTorch it is a single optimized function, [`F.scaled_dot_product_attention`](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html) (SDPA), used by most HuggingFace models.

Why divide by $\sqrt{d_k}$? The variance of $q^\top k$ for random vectors grows linearly with $d_k$, and large scores make the softmax almost one-hot, with tiny gradients. Let's check it for 64 random keys.

In [ ]:
num_keys = 64
dims = [4, 16, 64, 256, 1024]
largest_weight = {"q·k": [], "q·k / sqrt(d)": []}
for d in dims:
    q, k = torch.randn(1000, d), torch.randn(1000, num_keys, d)
    scores = torch.einsum("bd,bnd->bn", q, k)  # 1000 queries, 64 scores each
    for label, s in [("q·k", scores), ("q·k / sqrt(d)", scores / math.sqrt(d))]:
        largest_weight[label].append(s.softmax(dim=-1).max(dim=-1).values.mean())

fig, ax = plt.subplots(figsize=(6, 3.5))
for label, values in largest_weight.items():
    ax.semilogx(dims, values, "o-", lw=2, label=label, base=2)
ax.axhline(1 / num_keys, color="gray", ls="--", label="uniform (1/64)")
ax.set(
    xlabel="dimension d of queries and keys",
    ylabel="largest attention weight",
    title=f"Softmax over {num_keys} random keys",
)
ax.legend()
plt.tight_layout()
plt.show()

#### Masks

A **mask** says which keys each query may look at:

* **padding mask** - shorter texts in a batch are padded with `[PAD]` tokens, which are never attended to; tokenizers return it as `attention_mask`
* **causal mask** - in a decoder (GPT-style LLM), token $t$ sees only tokens $\leq t$

In [ ]:
tokenizer = tokenizers["BERT (leaf-mt)"]
batch = tokenizer(
    ["A short sentence.", "A bit longer sentence, with padding."],
    padding=True,
    return_tensors="pt",
)
print("input_ids:\n", batch["input_ids"])
print("attention_mask:\n", batch["attention_mask"])

tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0])
padding_mask = batch["attention_mask"][0].bool()
n = len(tokens)
masks = {
    "encoder: padding mask": padding_mask[None, :].expand(n, n),
    "decoder: causal + padding mask": torch.tril(torch.ones(n, n, dtype=torch.bool))
    & padding_mask[None, :],
}
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, (title, mask) in zip(axes, masks.items()):
    ax.imshow(mask, cmap="Greens", vmin=0, vmax=1.3)
    ax.set_xticks(range(n), tokens, rotation=90)
    ax.set_yticks(range(n), tokens)
    ax.set(title=title, xlabel="key (attended to)", ylabel="query")
plt.tight_layout()
plt.show()

### 1.3 Attention maps and contextual embeddings

`mdbr-leaf-mt` is a small BERT with 6 layers of 12 heads, i.e. 72 attention maps. `output_attentions=True` returns them; this needs the plain ("eager") attention implementation, since SDPA never stores the attention matrix. In the sentence below, "it" refers to "animal": let's find the head where "it" attends to "animal" the most, and compare it with two other heads.

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained(ST_MODEL)
bert = AutoModel.from_pretrained(ST_MODEL, attn_implementation="eager").eval()

sentence = "The animal didn't cross the street because it was too tired."
inputs = bert_tokenizer(sentence, return_tensors="pt")
with torch.no_grad():
    outputs = bert(**inputs, output_attentions=True)

tokens = bert_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
# tuple of 6 layers, each (batch, heads, queries, keys) -> (layers, heads, n, n)
attentions = torch.stack(outputs.attentions)[:, 0]
it, animal = tokens.index("it"), tokens.index("animal")
layer, head = divmod(attentions[:, :, it, animal].argmax().item(), attentions.shape[1])
print(
    f"layer {layer}, head {head}: 'it' -> 'animal' weight {attentions[layer, head, it, animal]:.2f}"
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, (l, h) in zip(axes, [(layer, head), (0, 0), (5, 0)]):
    ax.imshow(attentions[l, h], cmap="viridis")
    ax.set_xticks(range(len(tokens)), tokens, rotation=90)
    ax.set_yticks(range(len(tokens)), tokens)
    ax.set_title(f"layer {l}, head {h}")
plt.suptitle("Attention weights (row = query token, column = key token)")
plt.tight_layout()
plt.show()

Heads specialize: some follow relations like coreference, others attend to neighbours, and many put most weight on `[CLS]` or `[SEP]`. This is the **attention sink** from the lecture: softmax weights sum to 1, so a head with nothing to read dumps its attention on a token that is always present.

In [ ]:
for l in range(len(attentions)):
    cls, sep = attentions[l, :, :, 0].mean(), attentions[l, :, :, -1].mean()
    print(
        f"layer {l}: {100 * cls:4.1f}% of attention on [CLS], {100 * sep:4.1f}% on [SEP]"
        f" (uniform would be {100 / len(tokens):.1f}%)"
    )

#### Contextual embeddings

The input embedding of "bank" is a row of the embedding table, the same in every sentence. After attention, the output embedding depends on the context:

In [ ]:
sentences = [
    "We sat on the bank of the river.",
    "The river bank was covered with grass.",
    "I deposited money at the bank.",
    "The bank approved my loan.",
]
static, contextual = [], []
for s in sentences:
    encoded = bert_tokenizer(s, return_tensors="pt")
    position = bert_tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]).index(
        "bank"
    )
    with torch.no_grad():
        contextual.append(bert(**encoded).last_hidden_state[0, position])
        static.append(bert.get_input_embeddings()(encoded["input_ids"][0, position]))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), layout="constrained")
for ax, vectors, title in [
    (axes[0], static, "input embeddings (embedding table)"),
    (axes[1], contextual, "output embeddings (last layer)"),
]:
    normalized = F.normalize(torch.stack(vectors), dim=1)
    similarity = (normalized @ normalized.T).numpy()
    ax.imshow(similarity, cmap="viridis", vmin=0, vmax=1)
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center")
    ax.set_xticks(range(len(sentences)), [f"s{i + 1}" for i in range(len(sentences))])
    ax.set_yticks(
        range(len(sentences)), [f"s{i + 1}: {s}" for i, s in enumerate(sentences)]
    )
    ax.set_title(f"'bank': cosine similarity of {title}", fontsize=10)
axes[1].set_yticks([])
plt.show()

### 1.4 Positions

Attention treats the input as a set, so "dog bites man" and "man bites dog" would look identical. Positions are added either as **absolute** vectors added to tokens (fixed sinusoids in the original Transformer, learned in BERT), or as **relative** information inside attention, most often **RoPE** (LLaMA, Qwen, Gemma, ModernBERT).

Below, the cosine similarity between position vectors. In both, nearby positions are similar: the learned embeddings rediscovered this structure from data. BERT's learned table ends at 512 positions, while RoPE works for any length (but is only good at lengths seen in training).

In [ ]:
def sinusoidal_positions(num_positions: int, d: int) -> torch.Tensor:
    position = torch.arange(num_positions)[:, None]
    frequency = 10000 ** (-torch.arange(0, d, 2) / d)
    pe = torch.zeros(num_positions, d)
    pe[:, 0::2] = torch.sin(position * frequency)
    pe[:, 1::2] = torch.cos(position * frequency)
    return pe


learned = bert.embeddings.position_embeddings.weight.detach()  # (512, 384)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, pe, title in [
    (axes[0], sinusoidal_positions(*learned.shape), "sinusoidal"),
    (axes[1], learned, "learned (BERT)"),
]:
    pe = F.normalize(pe, dim=1)
    im = ax.imshow(pe @ pe.T, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set(title=f"{title}: cosine similarity", xlabel="position", ylabel="position")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

for model_name in [ENCODER_MODEL, LLM_MODEL]:
    config = transformers.AutoConfig.from_pretrained(model_name)
    print(
        f"{model_name}: max length {config.max_position_embeddings}, RoPE {config.rope_parameters}"
    )

### 1.5 Encoders and decoders

* An **encoder** (BERT, ModernBERT) is bidirectional, trained with **masked language modeling**, and used for embeddings, classification and retrieval
* A **decoder** (GPT, LLaMA, Qwen) uses the causal mask, is trained with **next-token prediction**, and generates text

The [Ettin](https://huggingface.co/collections/jhu-clsp/encoders-vs-decoders-the-ettin-suite) encoder and decoder have the same architecture, size and training data, so they are a fair comparison. The `fill-mask` pipeline predicts the hidden token with the encoder; for the decoder, we take the next-token probabilities from the logits.

In [ ]:
fill_mask = pipeline("fill-mask", model=ENCODER_MODEL)
decoder_tokenizer = AutoTokenizer.from_pretrained(DECODER_MODEL)
decoder = AutoModelForCausalLM.from_pretrained(DECODER_MODEL).eval()

for left, right in [
    ("The capital of France is", "."),
    ("I went to the", "to deposit my paycheck."),
]:
    masked = f"{left} {fill_mask.tokenizer.mask_token} {right}"
    predictions = fill_mask(masked, top_k=5)
    print(f"encoder: {masked}")
    print("  " + ", ".join(f"{p['token_str']!r} {p['score']:.2f}" for p in predictions))

    # the decoder sees only the left context, after the beginning-of-sequence token
    inputs = decoder_tokenizer(
        decoder_tokenizer.bos_token + left,
        return_tensors="pt",
        add_special_tokens=False,
    )
    with torch.no_grad():
        logits = decoder(**inputs).logits  # (batch, tokens, vocabulary)
    top = logits[0, -1].softmax(dim=-1).topk(5)  # next token after the last position
    print(f"decoder: {left} ...")
    print(
        "  "
        + ", ".join(
            f"{decoder_tokenizer.decode(i)!r} {p:.2f}"
            for p, i in zip(top.values, top.indices)
        )
        + "\n"
    )

The encoder uses both sides, so "to deposit my paycheck" makes "bank" almost certain; the decoder must guess from "I went to the". But only the decoder can generate: predict a token, append it, repeat, as `generate` does. With `do_sample=False` it always takes the most probable token (greedy decoding), and small base models then often loop. Instruction-tuned models in Part 2 behave much better.

In [ ]:
inputs = decoder_tokenizer(
    decoder_tokenizer.bos_token + "I went to the",
    return_tensors="pt",
    add_special_tokens=False,
)
output_ids = decoder.generate(**inputs, max_new_tokens=40, do_sample=False)
print(decoder_tokenizer.decode(output_ids[0], skip_special_tokens=True))

### 1.6 The cost of attention

Attention builds an `n x n` score matrix per head, so time and memory grow quadratically with the sequence length. Let's compare computing it "by hand" with the fused SDPA kernel, which doesn't store the whole matrix (FlashAttention and similar, on GPUs).

In [ ]:
def measure_attention(n: int, fused: bool, heads: int = 8, d: int = 64) -> float:
    q, k, v = torch.randn(3, 1, heads, n, d, device=device, dtype=HALF_DTYPE)
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(5):
        if fused:
            out = F.scaled_dot_product_attention(q, k, v)
        else:
            weights = (q @ k.transpose(-2, -1) / math.sqrt(d)).softmax(dim=-1)
            out = weights @ v
    out.sum().item()  # wait for the result
    return (time.perf_counter() - start) / 5


lengths = [256, 512, 1024, 2048, 4096, 8192]
for fused in (False, True):  # warmup
    measure_attention(256, fused)
times = {
    fused: [measure_attention(n, fused) for n in lengths] for fused in (False, True)
}
memory = [8 * n * n * torch.finfo(HALF_DTYPE).bits / 8 / 2**20 for n in lengths]

print(f"{'n':>5s} {'attention matrices (8 heads)':>29s} {'by hand':>9s} {'SDPA':>9s}")
for n, mb, t_naive, t_fused in zip(lengths, memory, times[False], times[True]):
    print(f"{n:5d} {mb:26.0f} MB {1000 * t_naive:7.1f}ms {1000 * t_fused:7.1f}ms")

#### KV cache

A decoder generates one token at a time. Without caching, every step recomputes keys and values of all previous tokens; the **KV cache** stores them, so each step computes only the new token. Let's turn it off, for an 800-token prompt.

In [ ]:
decoder = decoder.to(device)
inputs = decoder_tokenizer(
    decoder_tokenizer.bos_token + "Once upon a time " * 200,
    return_tensors="pt",
    add_special_tokens=False,
).to(device)
for use_cache in (True, False):
    start = time.perf_counter()
    decoder.generate(
        **inputs,
        max_new_tokens=200,
        min_new_tokens=200,
        do_sample=False,
        use_cache=use_cache,
    )
    print(f"use_cache={use_cache}: 200 tokens in {time.perf_counter() - start:.1f} s")

In [ ]:
# release the models (setting to None works like del here)
bert = fill_mask = decoder = None
free_memory()

# Part 2: Using pretrained Transformers

A pretrained Transformer turns a text into an **embedding**: texts with similar meaning get similar vectors. This allows semantic search, visualization and classification with simple models. We can also **fine-tune** the whole network, and combine embedding-based search with a generative LLM to answer questions about new documents.

### MASSIVE dataset

[MASSIVE](https://huggingface.co/datasets/mteb/amazon_massive_intent) (Amazon, 2022) contains short voice assistant commands, like "wake me up at nine am on friday", labeled with one of 60 **intents** (`alarm_set`, `weather_query`, ...). Intents are grouped into 18 **scenarios**, given by the first part of the name. There are 11.5k training and 3k test commands. Many intents are close (`alarm_set` vs `alarm_query`), so keyword matching is not enough. We load it with the [`datasets`](https://huggingface.co/docs/datasets/index) library.

In [ ]:
massive = load_dataset("mteb/amazon_massive_intent", "en")
train_texts, test_texts = massive["train"]["text"], massive["test"]["text"]
intent_names = sorted(set(massive["train"]["label_text"]))
y_train = np.array([intent_names.index(i) for i in massive["train"]["label_text"]])
y_test = np.array([intent_names.index(i) for i in massive["test"]["label_text"]])

# "alarm_set" -> "alarm"
scenario_names = sorted({i.split("_")[0] for i in intent_names})
scenario_of_intent = np.array(
    [scenario_names.index(i.split("_")[0]) for i in intent_names]
)
print(f"{len(train_texts)} train, {len(test_texts)} test texts")
print(f"{len(intent_names)} intents in {len(scenario_names)} scenarios\n")

rng = np.random.default_rng(0)
for i in rng.choice(len(train_texts), 8, replace=False):
    print(f"{intent_names[y_train[i]]:22s} {train_texts[i]}")

## 2.1 Sentence Transformers

Averaging token embeddings of a pretrained BERT gives surprisingly poor text embeddings. **Sentence Transformers** are encoders fine-tuned with contrastive learning on pairs of similar texts, so that their pooled output is a good embedding. The [`sentence-transformers`](https://sbert.net/) library does tokenization, batching and pooling inside `encode`.

We use [`mdbr-leaf-mt`](https://huggingface.co/MongoDB/mdbr-leaf-mt) (MongoDB, 2025), with only 23M parameters the best model of its size on the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard) when writing this. MTEB is the standard benchmark of embedding models, and a good place to choose one. The printed modules show: 384-dimensional token embeddings, mean pooling over real tokens, and a projection to 1024 dimensions (the model was distilled from a larger one).

In [ ]:
st_model = SentenceTransformer(ST_MODEL, device=device)
print(st_model)

With `normalize_embeddings=True` all embeddings have unit length, so dot product equals cosine similarity, which `st_model.similarity` computes for all pairs.

In [ ]:
sentences = [
    "set an alarm for seven tomorrow",
    "wake me up at 7 am",
    "will it rain in krakow today",
    "do I need an umbrella this afternoon",
    "the dog bit the man",
    "the man bit the dog",
]
embeddings = st_model.encode(sentences, normalize_embeddings=True)
similarity = st_model.similarity(embeddings, embeddings).numpy()

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.imshow(similarity, cmap="viridis", vmin=0, vmax=1)
for i in range(len(sentences)):
    for j in range(len(sentences)):
        color = "k" if similarity[i, j] > 0.7 else "w"
        ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center", color=color)
ax.set_xticks(range(len(sentences)), sentences, rotation=45, ha="right")
ax.set_yticks(range(len(sentences)), sentences)
ax.set_title(f"Cosine similarity of sentence embeddings {embeddings.shape}")
plt.tight_layout()
plt.show()

Paraphrases without common words get high similarity, but "dog bit man" and "man bit dog" are almost identical: mean pooling dilutes word order. Embeddings capture topic and meaning well, but not every detail.

#### Semantic textual similarity

[STS Benchmark](https://huggingface.co/datasets/sentence-transformers/stsb) has sentence pairs with human similarity scores from 0 to 1. The standard metric is the Spearman rank correlation between cosine similarities and human scores.

In [ ]:
from scipy.stats import spearmanr

stsb = load_dataset("sentence-transformers/stsb", split="test")
sentences1, sentences2 = stsb["sentence1"], stsb["sentence2"]
human_scores = np.array(stsb["score"])

embeddings1 = st_model.encode(sentences1, normalize_embeddings=True)
embeddings2 = st_model.encode(sentences2, normalize_embeddings=True)
# similarity of corresponding pairs only, not the full matrix
cosine_similarities = (embeddings1 * embeddings2).sum(axis=1)
sts_spearman = spearmanr(cosine_similarities, human_scores).statistic

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(human_scores, cosine_similarities, s=4, alpha=0.4)
ax.set(
    xlabel="human score",
    ylabel="cosine similarity",
    title=f"STS Benchmark ({len(stsb)} pairs), Spearman {sts_spearman:.3f}",
)
plt.tight_layout()
plt.show()

## 2.2 Embeddings and semantic search

We embed MASSIVE once (seconds on a GPU, below a minute on a CPU) and save it to disk.

In [ ]:
def cached_embeddings(path: str, compute: Callable) -> tuple[np.ndarray, np.ndarray]:
    # compute train and test embeddings once, then load them from disk
    path = os.path.join(DATA_ROOT, path)
    if not os.path.exists(path):
        start = time.time()
        np.savez(path, X_train=compute(train_texts), X_test=compute(test_texts))
        print(f"embeddings computed in {time.time() - start:.0f} s")
    saved = np.load(path)
    return saved["X_train"], saved["X_test"]


X_train, X_test = cached_embeddings(
    "massive_leaf_mt.npz",
    lambda texts: st_model.encode(texts, batch_size=64, normalize_embeddings=True),
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

**Semantic search** finds texts similar in meaning, not keywords: with normalized embeddings, it's a single matrix-vector product. Vector databases (Pinecone, Qdrant, pgvector, ...) do the same, with indexes for millions of vectors.

In [ ]:
def most_similar(
    query: np.ndarray, gallery: np.ndarray, k: int = 5
) -> tuple[np.ndarray, np.ndarray]:
    # query (D,), gallery (N, D), both normalized; returns top k indices and similarities
    similarities = gallery @ query
    indices = np.argsort(-similarities)[:k]
    return indices, similarities[indices]


queries = [
    "could you get me up early tomorrow",
    "is it going to be cold in krakow",
    "I'm hungry, find me a pizza place nearby",
    "tell me something funny",
]
query_embeddings = st_model.encode(queries, normalize_embeddings=True)
for query, query_embedding in zip(queries, query_embeddings):
    print(f"QUERY: {query}")
    for i, s in zip(*most_similar(query_embedding, X_train, k=4)):
        print(f"  {s:.2f} {intent_names[y_train[i]]:22s} {train_texts[i]}")

## 2.3 Visualizing the embedding space

As in Lab 1, t-SNE maps the test embeddings to 2D, colored by scenario. The model never saw MASSIVE, but alarms, weather or music form clear clusters. Scenarios like `general` (jokes, greetings) or `qa` (general questions) are not about one topic and mix.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# PCA to 50 dimensions first is standard: it is faster and removes noise
X_pca = PCA(n_components=50, random_state=0).fit_transform(X_test)
xy_test = TSNE(n_components=2, init="pca", random_state=0).fit_transform(X_pca)

fig, ax = plt.subplots(figsize=(10, 7.5))
scenarios = scenario_of_intent[y_test]
colors = plt.get_cmap("tab20").colors
for s, name in enumerate(scenario_names):
    mask = scenarios == s
    ax.scatter(*xy_test[mask].T, s=3, color=colors[s], label=name)
    # scenario name in the middle of its points
    ax.annotate(name, np.median(xy_test[mask], axis=0), fontsize=8, weight="bold")
ax.set_title(f"t-SNE of {ST_MODEL} embeddings (test set)")
ax.axis("off")
ax.legend(markerscale=4, loc="center left", bbox_to_anchor=(1, 0.5), title="scenario")
plt.tight_layout()
plt.show()

## 2.4 Linear probing

Embeddings turn text into a **tabular** dataset: one row per text, 1024 features and a label, ready for any scikit-learn model. A linear classifier on frozen embeddings is a **linear probe**: cheap, needs little data, and a standard measure of representation quality.

### Exercise 1: Linear probe

Train a [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) on `X_train` and `y_train` with `C=10` (weak regularization works well for normalized embeddings) and `max_iter=1000`, and compute its test accuracy, `probe_accuracy`. In a real project, you would select `C` with cross-validation.

In [ ]:
from sklearn.linear_model import LogisticRegression

probe = ...
probe_accuracy = ...
print(f"linear probe accuracy: {100 * probe_accuracy:.1f}%")


In [ ]:
assert isinstance(probe, LogisticRegression), "use LogisticRegression"
assert isinstance(probe_accuracy, float)
assert probe_accuracy > 0.83, (
    f"accuracy should be above 83%, got {100 * probe_accuracy:.1f}%"
)
print("OK")

How many labels do we need? Let's train the probe on 1 to 50 examples per intent. Even a handful gives a usable classifier, which is how many chatbots are built: a few example phrases per intent and a classifier on top of embeddings.

In [ ]:
# with 1 example per intent, scikit-learn warns that there are many classes for few samples
warnings.filterwarnings("ignore", message="The number of unique classes")

shots, few_shot_accuracy = [1, 2, 5, 10, 20, 50], []
for n in shots:
    # n random training examples of every intent (or all, if there are fewer)
    subset = np.concatenate(
        [
            rng.permutation(np.flatnonzero(y_train == c))[:n]
            for c in range(len(intent_names))
        ]
    )
    model = LogisticRegression(C=10, max_iter=1000).fit(
        X_train[subset], y_train[subset]
    )
    few_shot_accuracy.append(model.score(X_test, y_test))

plt.figure(figsize=(6, 3.5))
plt.semilogx(shots, 100 * np.array(few_shot_accuracy), "o-", lw=2)
plt.axhline(100 * probe_accuracy, color="gray", ls="--", label="all training data")
plt.xticks(shots, shots)
plt.xlabel("labelled training examples per intent")
plt.ylabel("test accuracy (%)")
plt.title("Linear probe on frozen embeddings")
plt.legend()
plt.show()

## 2.5 LLM embeddings: last-token pooling

The best embedding models today are often built from decoder LLMs, pretrained on much more data than encoders. Because of the causal mask, only the **last token** has seen the whole text, so its hidden state is used as the embedding. The LLM is then fine-tuned contrastively, like a Sentence Transformer.

We use [Harrier](https://huggingface.co/microsoft/harrier-oss-v1-270m) (Microsoft, 2026), built from Gemma 3 270M. Most of its parameters are the embedding table of its huge vocabulary (Part 1); the Transformer itself has about 100M. Gemma-based models overflow (NaN) in float16, so we use bfloat16 or float32.

In [ ]:
llm_embedding_dtype = torch.bfloat16 if HALF_DTYPE == torch.bfloat16 else torch.float32
harrier_tokenizer = AutoTokenizer.from_pretrained(LLM_EMBEDDING_MODEL)
harrier = AutoModel.from_pretrained(LLM_EMBEDDING_MODEL, dtype=llm_embedding_dtype)
harrier = harrier.to(device).eval()

### Exercise 2: Last-token pooling

By default, tokenizers pad on the right, so the last real token is at a different position in every row. With **left padding**, the last token of every text is at position `-1`, and the masks make sure padding changes nothing. The tokenizer also appends `<eos>`, so the last token is always `<eos>`.

Implement `last_token_embeddings`:

1. Tokenize `texts` with `harrier_tokenizer` with `padding=True`, `padding_side="left"`, `truncation=True`, `max_length=512`, `return_tensors="pt"`, and move the result to `device`
2. Run `harrier` without gradients; its output has `last_hidden_state` of shape `(batch, tokens, d)`
3. Take the last token's hidden state, convert it to float32 (`.float()`), normalize it with [`F.normalize`](https://pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html), and return a NumPy array on the CPU

In [ ]:
def last_token_embeddings(texts: list[str]) -> np.ndarray:
    ...


In [ ]:
check_texts = ["hi", "set an alarm for seven tomorrow morning please", "olly stop"]
ours = last_token_embeddings(check_texts)
assert isinstance(ours, np.ndarray) and ours.shape == (3, harrier.config.hidden_size)
assert np.allclose(np.linalg.norm(ours, axis=1), 1, atol=1e-3), (
    "normalize the embeddings"
)
# each text alone, without padding
single = np.concatenate([last_token_embeddings([t]) for t in check_texts])
assert np.allclose(ours, single, atol=0.05), (
    "padding changed the embeddings, did you use left padding and the last token?"
)
# the same model in sentence-transformers, which uses the same pooling
st_harrier = SentenceTransformer(
    LLM_EMBEDDING_MODEL, device=device, model_kwargs={"dtype": llm_embedding_dtype}
)
assert np.allclose(ours, st_harrier.encode(check_texts), atol=0.05), (
    "results differ from sentence-transformers"
)
st_harrier = None
print("OK")

Now we'll embed MASSIVE with Harrier.

In [ ]:
def embed_in_batches(texts: list[str], batch_size: int = 64) -> np.ndarray:
    return np.concatenate(
        [
            last_token_embeddings(texts[i : i + batch_size])
            for i in range(0, len(texts), batch_size)
        ]
    )


X_train_llm, X_test_llm = cached_embeddings("massive_harrier.npz", embed_in_batches)
llm_probe_accuracy = (
    LogisticRegression(C=10, max_iter=1000)
    .fit(X_train_llm, y_train)
    .score(X_test_llm, y_test)
)
print(
    f"linear probe: {100 * llm_probe_accuracy:.1f}% (Harrier) vs {100 * probe_accuracy:.1f}% (leaf-mt)"
)

#### Instructions

LLM-based embedders are trained with **instructions**: a task description prepended to the query, so the same text gets a different embedding for similarity, retrieval or classification. Let's check STS Benchmark with and without it.

In [ ]:
sts_instruction = "Instruct: Retrieve semantically similar text\nQuery: "
for instruction in ["", sts_instruction]:
    e1 = embed_in_batches([instruction + s for s in sentences1])
    e2 = embed_in_batches([instruction + s for s in sentences2])
    spearman = spearmanr((e1 * e2).sum(axis=1), human_scores).statistic
    print(f"instruction {instruction!r}: Spearman {spearman:.3f}")
print(f"leaf-mt (no instruction): Spearman {sts_spearman:.3f}")

harrier = None
free_memory()

Without its instruction, the LLM embedder loses to the much smaller Sentence Transformer; with it, it wins. Always check the model card for the expected prompts!

## 2.6 Fine-tuning

A linear probe can't change the embeddings. **Fine-tuning** adds a classification head and trains the whole network. We use `AutoModelForSequenceClassification` and the HuggingFace [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer), a ready-made training loop. We fine-tune our Sentence Transformer backbone for 5 epochs. `DataCollatorWithPadding` pads every batch only to its longest text.

In [ ]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

FINETUNE_EPOCHS = 5
FINETUNE_BATCH_SIZE = 32

ft_tokenizer = AutoTokenizer.from_pretrained(ST_MODEL)


def tokenize(batch: dict) -> dict:
    encoded = ft_tokenizer(batch["text"], truncation=True)
    encoded["labels"] = [intent_names.index(i) for i in batch["label_text"]]
    return encoded


massive_tokenized = massive.map(
    tokenize, batched=True, remove_columns=massive["train"].column_names
)
print(massive_tokenized["train"][0])

In [ ]:
transformers.set_seed(0)
classifier = AutoModelForSequenceClassification.from_pretrained(
    ST_MODEL,
    num_labels=len(intent_names),
    id2label=dict(enumerate(intent_names)),
    label2id={name: i for i, name in enumerate(intent_names)},
)
args = TrainingArguments(
    output_dir=os.path.join(DATA_ROOT, "finetuning"),
    num_train_epochs=FINETUNE_EPOCHS,
    learning_rate=2e-4,
    per_device_train_batch_size=FINETUNE_BATCH_SIZE,
    per_device_eval_batch_size=128,
    warmup_steps=0.1,  # 10% of the training steps
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    # mixed precision on CUDA GPUs
    bf16=device.type == "cuda" and HALF_DTYPE == torch.bfloat16,
    fp16=device.type == "cuda" and HALF_DTYPE == torch.float16,
)


def compute_metrics(eval_prediction: transformers.EvalPrediction) -> dict:
    predictions = eval_prediction.predictions.argmax(axis=-1)
    return {"accuracy": (predictions == eval_prediction.label_ids).mean()}


trainer = Trainer(
    model=classifier,
    args=args,
    train_dataset=massive_tokenized["train"],
    eval_dataset=massive_tokenized["test"],
    data_collator=DataCollatorWithPadding(ft_tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()
finetune_accuracy = trainer.evaluate()["eval_accuracy"]

The fine-tuned model is a regular HuggingFace model, usable e.g. in a `text-classification` pipeline.

In [ ]:
classify = pipeline(
    "text-classification", model=classifier, tokenizer=ft_tokenizer, device=device
)
for text in [
    "could you get me up early tomorrow",
    "turn the lights in the kitchen off",
    "how much is a bitcoin worth in dollars",
]:
    prediction = classify(text)[0]
    print(f"{prediction['label']:22s} {prediction['score']:.2f}  {text}")

results = {
    "linear probe, leaf-mt (23M)": probe_accuracy,
    "linear probe, Harrier (270M)": llm_probe_accuracy,
    "fine-tuning, leaf-mt (23M)": finetune_accuracy,
}
plt.figure(figsize=(7, 2.4))
bars = plt.barh(
    list(results)[::-1],
    [100 * a for a in list(results.values())[::-1]],
    color="#7d6cc4",
)
plt.bar_label(bars, fmt="%.1f%%", padding=3)
plt.xlim(0, 100)
plt.xlabel("test accuracy (%)")
plt.title("Intent classification on MASSIVE (60 intents)")
plt.show()

classifier = trainer = classify = None
free_memory()

Fine-tuning the small model beats probing a much larger one. The gap between probing and fine-tuning is much smaller than for birds in Lab 1: general-purpose text embeddings already capture what this task needs.

## 2.7 Text generation with an LLM

[Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) (Alibaba, 2025) is a small but capable decoder LLM, about 3.5 GB in 16-bit precision. It generates dozens of tokens per second on a GPU and a few on a CPU.

Chat LLMs expect a list of **messages** with a `role` (`system`, `user` or `assistant`) and `content`, like the OpenAI or Anthropic APIs. The tokenizer's **chat template** turns it into a string with special tokens; `add_generation_prompt=True` adds the start of the assistant's turn for the model to continue. Qwen3 is a **reasoning model**: by default it first "thinks" inside `<think> ... </think>`, which helps on hard questions but costs many tokens. We turn it off with `enable_thinking=False` by default.

In [ ]:
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
llm = (
    AutoModelForCausalLM.from_pretrained(LLM_MODEL, dtype=HALF_DTYPE).to(device).eval()
)

messages = [
    {"role": "user", "content": "What is a transformer? Answer in two sentences."}
]
print(
    llm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
)

In [ ]:
def chat(
    messages: list[dict],
    max_new_tokens: int = 256,
    thinking: bool = False,
    temperature: float | None = None,
) -> str:
    inputs = llm_tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=thinking,
        return_tensors="pt",
        return_dict=True,
    ).to(device)
    output_ids = llm.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature is not None,
        temperature=temperature,
    )
    # decode only the new tokens, without the prompt
    new_ids = output_ids[0, inputs["input_ids"].shape[1] :]
    return llm_tokenizer.decode(new_ids, skip_special_tokens=True).strip()


print(chat(messages))

The question was ambiguous, so the model picked the most common meaning. A **system prompt** sets the context and style:

In [ ]:
system_prompt = (
    "You are a teaching assistant in a university course on neural network "
    "architectures. Answer briefly and precisely."
)
print(chat([{"role": "system", "content": system_prompt}] + messages))

With sampling, the next token is drawn from the predicted distribution. The **temperature** divides the logits before the softmax: low values make it sharper (more deterministic), high values flatter (more diverse).

In [ ]:
story = [
    {"role": "user", "content": "Write a one-sentence story about a robot and a cat."}
]
for temperature in [0.3, 0.3, 1.2, 1.2]:
    print(f"temperature {temperature}: {chat(story, temperature=temperature)}")

Finally, reasoning on a famous trick question. LLMs see tokens, not letters, so counting letters is hard for them. On a CPU this cell may take a few minutes.

In [ ]:
question = [
    {"role": "user", "content": "How many letters 'r' are in the word 'strawberry'?"}
]
print("tokens:", llm_tokenizer.tokenize(" strawberry"))
for thinking in (False, True):
    answer = chat(question, max_new_tokens=1024, thinking=thinking)
    print(
        f"\n=== thinking={thinking}, {len(llm_tokenizer.tokenize(answer))} tokens:\n{answer}"
    )

## 2.8 Retrieval-augmented generation (RAG)

An LLM knows only its training data, and confidently makes up answers about everything else ("hallucinations"). **RAG** fixes this with semantic search:

1. split the documents into **chunks** and embed them, once
2. embed the question and **retrieve** the most similar chunks
3. put them into the prompt and let the LLM **answer** based on them

Our documents are the [EU AI Act](https://eur-lex.europa.eu/eli/reg/2024/1689/oj) (2024), the European regulation of AI, from [a HuggingFace dataset](https://huggingface.co/datasets/jeroenherczeg/eu-ai-act) of recitals, article paragraphs and annex points, about 90k words. Long fragments (e.g. Article 3 with 68 definitions) would be truncated by the embedding model, so we split them into chunks of up to 200 words, prefixed with the citation label for the LLM to cite.

In [ ]:
ai_act = load_dataset("jeroenherczeg/eu-ai-act", split="train")
# English only; skip whole articles, since their paragraphs are also included separately
ai_act = ai_act.filter(
    lambda row: row["language"] == "en" and row["chunk_type"] != "article_full"
)


def chunk_text(text: str, max_words: int = 200) -> list[str]:
    # greedily pack whole lines into chunks of at most max_words words
    chunks, current = [], []
    for line in text.split("\n"):
        words = line.split()
        if current and len(current) + len(words) > max_words:
            chunks.append(" ".join(current))
            current = []
        current += words
    if current:
        chunks.append(" ".join(current))
    return chunks


chunk_ids, chunks = [], []
for row in ai_act:
    for i, chunk in enumerate(chunk_text(row["text"])):
        chunk_ids.append(f"{row['id']}#{i}")  # e.g. "art-5-par-1-en#0"
        chunks.append(f"{row['citation_label']}: {chunk}")
print(f"{len(ai_act)} fragments -> {len(chunks)} chunks, for example:")
for i in [0, 330, 790]:
    print(f"  {chunks[i][:200]}...")

chunk_embeddings = st_model.encode(chunks, batch_size=32, normalize_embeddings=True)
print(f"chunk embeddings: {chunk_embeddings.shape}")

### Exercise 3: RAG

1. Implement `retrieve`, which returns the indices of the `k` chunks most similar to `question`, from the most similar one. Questions and documents are different kinds of text, so `mdbr-leaf-mt` has a special prompt for queries: encode the question with `prompt_name="query"` and `normalize_embeddings=True` (documents were encoded without a prompt). Then use `most_similar` from section 2.2
2. Implement `build_rag_messages`, which returns the messages for the LLM: a **system message** with `RAG_INSTRUCTIONS`, and a **user message** with the chunks separated by empty lines, followed by the question:

```
Context:
Art. 99(6) AI Act: In the case of SMEs, including start-ups, each fine ...

Art. 99(1) AI Act: In accordance with the terms and conditions laid down ...

Question: Are fines lower for small companies and start-ups?
```

In [ ]:
RAG_INSTRUCTIONS = (
    "You answer questions about the EU AI Act. Answer only based on the context "
    "below, briefly, and cite the articles, recitals or annexes you used. If the "
    "context does not contain the answer, say that you don't know."
)


def retrieve(question: str, k: int = 5) -> np.ndarray:
    ...


def build_rag_messages(question: str, context_chunks: list[str]) -> list[dict]:
    ...


In [ ]:
indices = retrieve("Are fines lower for small companies and start-ups?", k=3)
assert len(indices) == 3, "return k indices"
assert chunk_ids[indices[0]].startswith("art-99-par-6"), (
    "Article 99(6) (fines for SMEs) should be the top result"
)
assert len(set(retrieve("What is a deep fake?", k=10).tolist())) == 10, (
    "return k different chunks"
)

messages = build_rag_messages("Q?", ["chunk A", "chunk B"])
assert [m["role"] for m in messages] == ["system", "user"]
assert messages[0]["content"] == RAG_INSTRUCTIONS
user_content = messages[1]["content"]
assert "chunk A" in user_content and "chunk B" in user_content and "Q?" in user_content
assert (
    user_content.index("chunk A")
    < user_content.index("chunk B")
    < user_content.index("Q?")
), "put the chunks in the given order, then the question"
print("OK")

The whole RAG system is now a few lines. Let's compare answers without and with retrieval.

In [ ]:
def answer_with_rag(question: str, k: int = 5) -> str:
    indices = retrieve(question, k)
    print("retrieved:", ", ".join(chunks[i].split(":")[0] for i in indices))
    return chat(build_rag_messages(question, [chunks[i] for i in indices]))


for question in [
    "What is the maximum fine for using a prohibited AI practice?",
    "Is it allowed to use AI to recognize the emotions of employees at work?",
    "Does the AI Act apply to AI systems released under free and open-source licences?",
]:
    no_rag_messages = [
        {"role": "system", "content": "Answer briefly, in at most three sentences."},
        {"role": "user", "content": question},
    ]
    print(f"QUESTION: {question}")
    print(f"WITHOUT RAG: {chat(no_rag_messages)}")
    print(f"WITH RAG: {answer_with_rag(question)}\n")

Without context, the model gives generic or wrong answers. With retrieval, it gives exact amounts, exceptions and articles. Still verify the citations: small LLMs sometimes misread the context.

#### Evaluating retrieval

The answer is only as good as the retrieved context. Below are hand-written questions with known relevant ("gold") fragments. **Recall@k** is the fraction of questions with a relevant fragment among the top $k$ chunks.

In [ ]:
# (question, ids of relevant fragments, any matching prefix counts)
eval_questions = [
    ("What is the maximum fine for using a prohibited AI practice?", ["art-99-par-3"]),
    (
        "When do the obligations for providers of general-purpose AI models start to apply?",
        ["art-113-par"],
    ),
    (
        "Does the AI Act apply to AI systems released under free and open-source licences?",
        ["art-2-par-12"],
    ),
    (
        "Do chatbots have to tell people that they are talking to an AI?",
        ["art-50-par-1", "rec-132"],
    ),
    (
        "Do deep fakes have to be labelled as artificially generated?",
        ["art-50-par-4", "rec-134"],
    ),
    (
        "Is it allowed to use AI to recognize the emotions of employees at work?",
        ["art-5-par-1", "rec-44"],
    ),
    (
        "Is an AI system that filters job applications considered high-risk?",
        ["anx-III-sec-4", "rec-57"],
    ),
    (
        "Does the AI Act apply to me if I use AI only for personal, non-professional purposes?",
        ["art-2-par-10"],
    ),
    (
        "What must companies do about the AI literacy of their staff?",
        ["art-4-par", "rec-20"],
    ),
    ("Are fines lower for small companies and start-ups?", ["art-99-par-6"]),
    (
        "Does the regulation apply to AI developed only for scientific research?",
        ["art-2-par-6", "rec-25"],
    ),
    ("Is social scoring of people by AI prohibited?", ["art-5-par-1", "rec-31"]),
    ("How is a 'deployer' defined?", ["art-3-par"]),
    ("Which AI systems used in education are high-risk?", ["anx-III-sec-3", "rec-56"]),
]

max_k = 10
hits = np.zeros((len(eval_questions), max_k), dtype=bool)
for q, (question, gold) in enumerate(eval_questions):
    for rank, i in enumerate(retrieve(question, k=max_k)):
        hits[q, rank] = any(chunk_ids[i].startswith(g) for g in gold)

for k in (1, 3, 5, 10):
    print(f"recall@{k:<2d}: {100 * hits[:, :k].any(axis=1).mean():.0f}%")
print("\nmissed in top 5:")
for q in np.flatnonzero(~hits[:, :5].any(axis=1)):
    print(f"  {eval_questions[q][0]}")

A failure: the dates for general-purpose AI models are in Article 113, "Chapter V ... shall apply from 2 August 2025". Chapter V is about general-purpose AI models, but the fragment never says so, and semantic search can't find it. Without the right context, the LLM can't answer correctly either:

In [ ]:
print(answer_with_rag(eval_questions[1][0]))
print("\nArticle 113:", [c for c in chunks if c.startswith("Art. 113")])

Real RAG systems improve on this with better chunking (e.g. adding chapter titles to chunks), **hybrid search** (semantic + keyword search like BM25), **reranking** the top results with a slower, more accurate cross-encoder, or letting the LLM decide what to search for (**agentic search**). The core stays the same: embeddings find the relevant text, the LLM reads it.

In [ ]:
# free the memory before Part 3
llm = st_model = None
free_memory()

# Part 3: From BERT to ModernBERT

Now we build a Transformer encoder ourselves and modernize it step by step, as in Part 3 of lab 1: from a BERT-style encoder (2018) to the improvements used in [ModernBERT](https://arxiv.org/abs/2412.13663) (2024) and in practically all current LLMs. Each step adds one change, which you implement, check and train. The models are small (about 5M parameters) and train from scratch in 1-3 minutes per step on a GPU (Colab T4 is enough).

| Step | Change | You implement |
|---|---|---|
| 0 | BPE tokenizer for RNA | Exercise 4: tokenizer |
| 1 | BERT-style encoder: learned positions, LayerNorm, GELU MLP, [CLS] token | Exercises 5-6: Transformer block, encoder |
| 2 | rotary positional embeddings (RoPE) | Exercise 7: RoPE |
| 3 | gated MLP (GeGLU) | Exercise 8: GeGLU |
| 4 | mean pooling | Exercise 9: mean pooling |
| 5 | FlashAttention and unpadding | Exercise 10: unpadded encoder |
| 6 | sliding window + global attention | Exercise 11: sliding window encoder |

Steps 1-4 change the architecture and should improve accuracy. Steps 5-6 are about efficiency: (almost) the same function, computed faster or for longer sequences.

**Dataset.** Instead of text, where pretrained LLMs are hard to beat, we'll classify RNA sequences on [nRC](https://academic.oup.com/bioinformatics/article/33/17/2589/3836446) dataset. It contains non-coding RNA from 13 families (tRNA, rRNA, microRNA, riboswitches, ...), part of the [BEACON](https://arxiv.org/abs/2406.10391) benchmark. RNA is a sequence of nucleotides A, C, G and U, here 40-1200 long. The family depends on how the molecule folds, i.e. which distant parts of the sequence pair with each other, a natural task for attention. Simple baselines: the length alone gives 46% accuracy, logistic regression on 5-mer counts 70%.

**What to expect.** Steps 2-4 will improve accuracy bit by bit. Seeds are fixed, but some GPU kernels are not deterministic, so results vary between runs by up to about 1 point: look at the overall trend rather than single steps. The efficiency steps keep the accuracy or slightly reduce it, but change the training time a lot.

## Settings, data, and training tools

### Data

The sequences are given as DNA (T instead of U), we convert them to RNA. There are 5670 training, 650 validation and 2600 test sequences, with 450 training examples per class (270 for one class).

In [ ]:
EPOCHS = 30
BATCH_SIZE = 64

nrc = load_dataset("jiahaozhang2003/beacon-noncoding-rna-family")
rna = {
    split: [s.upper().replace("T", "U") for s in nrc[split]["sequence"]]
    for split in ("train", "validation", "test")
}
rna_labels = {
    split: np.array(nrc[split]["label"]) for split in ("train", "validation", "test")
}
NUM_CLASSES = int(rna_labels["train"].max()) + 1

lengths = np.array([len(s) for s in rna["train"]])
print(
    f"{len(rna['train'])} train / {len(rna['validation'])} val / {len(rna['test'])} test sequences, "
    f"{NUM_CLASSES} classes, length {lengths.min()}-{lengths.max()} nt (median {np.median(lengths):.0f})"
)
print("example:", rna["train"][0])

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lengths, bins=100)
ax.set(
    xlabel="sequence length (nucleotides)", ylabel="sequences", title="nRC training set"
)
plt.tight_layout()
plt.show()

## Step 0: Tokenizer

Single nucleotides as tokens give long sequences, up to 1182 tokens, and attention cost grows quadratically with length. As for text, we train a BPE tokenizer (Part 1), which merges frequent substrings of nucleotides into single tokens, e.g. parts of conserved motifs. A vocabulary of 256 tokens (about 3.4 nucleotides per token) was the best in our experiments: as accurate as single nucleotides and 3-4x faster; with 1024 tokens, the accuracy drops by a few points.

Special tokens are as in BERT: `[PAD]`, `[CLS]` at the start, `[SEP]` at the end, `[UNK]` for unknown characters (e.g. `N`, an unknown nucleotide) and `[MASK]` (unused here). `[PAD]` has id 0, so `ids != 0` is the padding mask.

### Exercise 4: BPE tokenizer

Train a BPE tokenizer with the HuggingFace [`tokenizers`](https://huggingface.co/docs/tokenizers/index) library (used by the fast tokenizers in `transformers`):
1. create a `Tokenizer` with a `models.BPE` model, with `unk_token="[UNK]"`
2. create a `trainers.BpeTrainer` with the `vocab_size` and `SPECIAL_TOKENS` (as `special_tokens`), in this order, so that `[PAD]` gets id 0
3. train the tokenizer with `train_from_iterator` on the training sequences

Then implement `encode`: tokenize sequences with `tokenizer.encode_batch`, and return token ids (`.ids` of every encoding) with `[CLS]` added at the start and `[SEP]` at the end.

Useful: [quicktour](https://huggingface.co/docs/tokenizers/quicktour), [`BpeTrainer`](https://huggingface.co/docs/tokenizers/api/trainers#tokenizers.trainers.BpeTrainer).

In [ ]:
SPECIAL_TOKENS = ["[PAD]", "[CLS]", "[SEP]", "[MASK]", "[UNK]"]
PAD, CLS, SEP = 0, 1, 2
VOCAB_SIZE = 256


def train_tokenizer(sequences: list[str], vocab_size: int) -> Tokenizer:
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    trainer = trainers.BpeTrainer(...)
    ...


def encode(tokenizer: Tokenizer, sequences: list[str]) -> list[list[int]]:
    ...

In [ ]:
rna_tokenizer = train_tokenizer(rna["train"], VOCAB_SIZE)
assert rna_tokenizer.get_vocab_size() == VOCAB_SIZE, "wrong vocabulary size"
assert [rna_tokenizer.token_to_id(t) for t in SPECIAL_TOKENS] == list(range(5)), (
    "special tokens should have ids 0-4, in the order of SPECIAL_TOKENS"
)
ids = encode(rna_tokenizer, ["ACGU", "GGGAAAUCC"])
assert all(x[0] == CLS and x[-1] == SEP for x in ids), "add [CLS] and [SEP]"
assert rna_tokenizer.decode(ids[1]).replace(" ", "") == "GGGAAAUCC"

rna_ids = {split: encode(rna_tokenizer, seqs) for split, seqs in rna.items()}
token_lengths = np.array([len(x) for x in rna_ids["train"]])
print(
    f"tokens per sequence: median {np.median(token_lengths):.0f}, max {token_lengths.max()}, "
    f"{(lengths.sum() / (token_lengths - 2).sum()):.2f} nucleotides per token"
)
encoding = rna_tokenizer.encode(rna["train"][0][:60])
print("tokens:", encoding.tokens)
vocab = sorted(rna_tokenizer.get_vocab(), key=len, reverse=True)
print("longest tokens:", vocab[:8])

print("OK")

### Batches

Shorter sequences in a batch are padded with `[PAD]` to the longest one. In random batches, one long sequence usually comes with many short ones, so most of the batch is padding.

In [ ]:
def make_batches(
    ids: list[list[int]],
    labels: np.ndarray,
    batch_size: int,
    shuffle: bool = False,
    seed: int = 0,
) -> Iterator[tuple[torch.Tensor, torch.Tensor]]:
    # yields (token ids (B, L) padded to the longest sequence in the batch, labels (B,))
    order = (
        np.random.default_rng(seed).permutation(len(ids))
        if shuffle
        else range(len(ids))
    )
    for start in range(0, len(ids), batch_size):
        batch = order[start : start + batch_size]
        x = torch.full((len(batch), max(len(ids[i]) for i in batch)), PAD)
        for row, i in enumerate(batch):
            x[row, : len(ids[i])] = torch.tensor(ids[i])
        yield x, torch.as_tensor(labels[batch])


x, y = next(make_batches(rna_ids["train"], rna_labels["train"], 8, shuffle=True))
print(x[:, :12])
batches = list(make_batches(rna_ids["train"], rna_labels["train"], BATCH_SIZE, True))
padding = 1 - sum((x != PAD).sum().item() for x, _ in batches) / sum(
    x.numel() for x, _ in batches
)
print(f"random batches: {100 * padding:.0f}% padding")

### Training tools

`fit` trains with the standard recipe of modern Transformers: AdamW, warmup and cosine decay of the learning rate, gradient clipping, mixed precision. `train_step` trains a model built by `make_model`, compares it with the previous step, saves the results, and plots the test accuracy, parameters and training time per epoch of all steps. `check_block` checks that a layer maps a padded batch `(B, L, dim)` with a padding mask `(B, L)` to `(B, L, dim)`.

This section is collapsed: you only need to run it.

In [ ]:
USE_AMP = device.type == "cuda"  # mixed precision on NVIDIA GPUs
RESULTS_PATH = os.path.join(DATA_ROOT, "modernbert_steps_results.json")
results = {}
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
    print(f"loaded results of {len(results)} steps from {RESULTS_PATH}")


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def autocast() -> torch.autocast:
    return torch.autocast(device_type=device.type, dtype=HALF_DTYPE, enabled=USE_AMP)


def synchronize() -> None:
    # wait until the GPU finishes all queued work, needed for correct timing
    if torch.accelerator.is_available():
        torch.accelerator.synchronize()


@torch.no_grad()
def evaluate(model: nn.Module, split: str) -> float:
    model.eval()
    correct = 0
    for x, y in make_batches(rna_ids[split], rna_labels[split], 256):
        with autocast():
            correct += (model(x.to(device)).argmax(dim=-1).cpu() == y).sum().item()
    return correct / len(rna_labels[split])


def fit(model: nn.Module, epochs: int = EPOCHS, lr: float = 1e-3) -> float:
    # trains the model, returns the median time per epoch (the first one is slower)
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = epochs * math.ceil(len(rna_ids["train"]) / BATCH_SIZE)
    scheduler = transformers.get_cosine_schedule_with_warmup(
        opt, int(0.05 * total_steps), total_steps
    )
    scaler = torch.amp.GradScaler(enabled=USE_AMP)  # avoids float16 underflow
    epoch_times = []
    for epoch in range(1, epochs + 1):
        model.train()
        start, loss_sum = time.time(), 0.0
        for x, y in make_batches(
            rna_ids["train"], rna_labels["train"], BATCH_SIZE, True, epoch
        ):
            x, y = x.to(device), y.to(device)
            with autocast():
                loss = F.cross_entropy(model(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            scheduler.step()
            loss_sum += loss.item() * len(y)
        synchronize()
        epoch_times.append(time.time() - start)
        if epoch % 5 == 0 or epoch == epochs:
            print(
                f"epoch {epoch:2d}/{epochs} | "
                f"train loss {loss_sum / len(rna_ids['train']):.3f} | "
                f"val accuracy {100 * evaluate(model, 'validation'):.1f}% | "
                f"{epoch_times[-1]:.1f} s"
            )
    return float(np.median(epoch_times))


def train_step(name: str, make_model: Callable[[], nn.Module]) -> nn.Module:
    transformers.set_seed(0)  # the same initialization and dropout for every step
    model = make_model()
    params = count_params(model) / 1e6
    print(f"{name}: {params:.2f}M parameters, {EPOCHS} epochs")
    epoch_time = fit(model)
    results[name] = dict(
        accuracy=evaluate(model, "test"), params=params, epoch_time=epoch_time
    )
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f)

    message = f"test accuracy {100 * results[name]['accuracy']:.1f}%"
    names = list(results)
    if (i := names.index(name)) > 0:
        old = results[names[i - 1]]
        message += (
            f", compared with '{names[i - 1]}': "
            f"{100 * (results[name]['accuracy'] - old['accuracy']):+.1f} points, "
            f"training time {100 * (epoch_time / old['epoch_time'] - 1):+.0f}%"
        )
    print(message)
    plot_results()
    return model


def plot_results() -> None:
    names = list(results)
    accuracy = [100 * results[n]["accuracy"] for n in names]
    fig, ax = plt.subplots(figsize=(9, 0.4 * len(names) + 1))
    bars = ax.barh(names, accuracy, color="#7d6cc4")
    labels = [
        f"{a:.1f}%  ({results[n]['params']:.1f}M params, {results[n]['epoch_time']:.1f} s/epoch)"
        for n, a in zip(names, accuracy)
    ]
    ax.bar_label(bars, labels, padding=4, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(max(0, min(accuracy) - 10), 100)
    ax.set_xlabel("nRC test accuracy (%)")
    plt.tight_layout()
    plt.show()


def check_block(block: nn.Module, dim: int, num_params: int | None = None) -> None:
    # shape contract: (B, L, dim) + padding mask (B, L) -> (B, L, dim)
    x = torch.randn(3, 10, dim)
    mask = torch.ones(3, 10, dtype=torch.bool)
    mask[1, 6:] = False
    y = block(x, mask)
    assert tuple(y.shape) == (3, 10, dim), (
        f"expected output shape {(3, 10, dim)}, got {tuple(y.shape)}"
    )
    if num_params is not None:
        got = count_params(block)
        assert got == num_params, f"expected {num_params} parameters, got {got}"
    print(f"{type(block).__name__}: OK, {count_params(block)} parameters")

## Step 1: BERT

The first model is a small [BERT](https://arxiv.org/abs/1810.04805)-style encoder: 6 layers of width 256 with 4 attention heads, about 5M parameters, built from the bottom up: the Transformer block, then the encoder. Unlike the original BERT, we use **pre-norm** blocks (normalization at the start of each residual branch), which train much more stably and are used by practically all modern Transformers.

### Attention

We use multi-head self-attention (Part 1, section 1.2) from PyTorch, [`nn.MultiheadAttention`](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html). It computes queries, keys and values with one linear layer (`dim -> 3 * dim`), splits them into heads, runs SDPA and merges the heads with an output projection. Our blocks pass a padding mask `(B, L)` with `True` for real tokens, while `nn.MultiheadAttention` takes `key_padding_mask` with `True` for padding, so we wrap it.

In [ ]:
class Attention(nn.Module):
    # PyTorch multi-head self-attention, with a padding mask (B, L), True for real tokens
    def __init__(self, dim: int, heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.mha = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        out, _ = self.mha(x, x, x, key_padding_mask=~mask, need_weights=False)
        return out

In [ ]:
attention = Attention(256, heads=4).eval()
check_block(attention, 256, num_params=4 * 256 * 256 + 4 * 256)

### Exercise 5: Transformer block

Implement the pre-norm Transformer block:

```
x -> x + dropout(attention(norm1(x), mask))
  -> x + dropout(mlp(norm2(x)))
```

The attention, MLP and normalization layers are created by `make_attention(dim)`, `make_mlp(dim)` and `make_norm(dim)`, so that the next steps can plug in other layers. The BERT MLP is given.

In [ ]:
class MLP(nn.Module):
    # BERT MLP: dim -> 4 * dim -> GELU -> dim
    def __init__(self, dim: int) -> None:
        super().__init__()
        self.up = nn.Linear(dim, 4 * dim)
        self.down = nn.Linear(4 * dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down(F.gelu(self.up(x)))


class Block(nn.Module):
    def __init__(
        self,
        dim: int,
        make_attention: Callable[[int], nn.Module],
        make_mlp: Callable[[int], nn.Module],
        make_norm: Callable[[int], nn.Module],
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.norm1 = ...
        self.attention = ...
        self.norm2 = ...
        self.mlp = ...
        self.dropout = ...

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        ...

In [ ]:
block = Block(256, partial(Attention, heads=4), MLP, nn.LayerNorm).eval()
check_block(block, 256, num_params=789760)
assert isinstance(block.norm1, nn.LayerNorm) and isinstance(block.mlp, MLP)

# with the output projections zeroed, both residual branches output 0: the block is an identity
for layer in (block.attention.mha.out_proj, block.mlp.down):
    nn.init.zeros_(layer.weight)
    nn.init.zeros_(layer.bias)
x = torch.randn(2, 5, 256)
assert torch.allclose(block(x, torch.ones(2, 5, dtype=torch.bool)), x), (
    "check the residual connections"
)

print("OK")

### Exercise 6: Encoder

Assemble the encoder:

* `self.embedding`: token embeddings `nn.Embedding(vocab_size, dim)`
* `self.positions`: learned position embeddings `nn.Embedding(max_len, dim)`, or `None` if `max_len` is `None` (from step 2)
* `self.dropout`: dropout of the input embeddings
* `self.blocks`: an `nn.ModuleList` of `depth` blocks
* `self.norm`: the final normalization, needed with pre-norm blocks
* `self.pooling`: created by `make_pooling(dim)`, by default the `[CLS]` token embedding
* `self.head`: the classification head, given

`forward`: padding mask (`ids != PAD`), token + position embeddings (positions `0, 1, ..., L - 1`), dropout, blocks, final norm, pooling and head.

In [ ]:
class CLSPooling(nn.Module):
    # the output embedding of the first token, [CLS]
    def __init__(self, dim: int) -> None:
        super().__init__()

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        return x[:, 0]


class Encoder(nn.Module):
    def __init__(
        self,
        make_attention: Callable[[int], nn.Module],
        make_mlp: Callable[[int], nn.Module] = MLP,
        make_norm: Callable[[int], nn.Module] = nn.LayerNorm,
        make_pooling: Callable[[int], nn.Module] = CLSPooling,
        vocab_size: int = VOCAB_SIZE,
        num_classes: int = NUM_CLASSES,
        dim: int = 256,
        depth: int = 6,
        max_len: int | None = 512,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.embedding = ...
        self.positions = ...
        self.dropout = ...
        self.blocks = ...
        self.norm = ...
        self.pooling = ...
        self.head = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, num_classes),
        )

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        ...

In [ ]:
torch.manual_seed(0)
model = Encoder(partial(Attention, heads=4, dropout=0.1)).eval()

assert len(model.blocks) == 6, "wrong number of blocks"
assert isinstance(model.norm, nn.LayerNorm), "add the final normalization"
ids = torch.tensor([[CLS, 10, 11, 12, SEP], [CLS, 20, SEP, PAD, PAD]])
assert model(ids).shape == (2, NUM_CLASSES), (
    "the output should be (batch size, classes)"
)
assert torch.allclose(model(ids[1:, :3]), model(ids)[1:], atol=1e-5), (
    "padding should not change the output"
)
assert not torch.allclose(model(ids[:1]), model(ids[:1, [0, 2, 1, 3, 4]])), (
    "the order of tokens should matter (positional embeddings)"
)
assert count_params(model) == 5004813, (
    f"expected 5004813 parameters, got {count_params(model)}"
)

print("OK")

In [ ]:
HEADS = 4
DROPOUT = 0.1


def make_model() -> nn.Module:
    return Encoder(partial(Attention, heads=HEADS, dropout=DROPOUT))


model = train_step("1. BERT", make_model)

## Step 2: Rotary positional embeddings (RoPE)

BERT's learned positions are absolute, while what usually matters is the relative position of two tokens (e.g. bases that pair in a hairpin), and they end at the size of the table (512 in BERT).

[RoPE](https://arxiv.org/abs/2104.09864) (Rotary Position Embedding) encodes positions inside attention instead: pairs of numbers in query and key vectors are rotated by the angle $m \theta_i$, where $m$ is the token position and $\theta_i = 10000^{-2i/d}$ the frequency of the pair (as in the sinusoidal embeddings from Part 1). The dot product of two rotated vectors depends only on the difference of their angles, so the attention score $q_m^\top k_n$ depends only on the relative position $m - n$. RoPE has no parameters and is used in ModernBERT and practically all modern LLMs.

### Our own attention

`nn.MultiheadAttention` computes queries and keys internally, so we can't rotate them. `SelfAttention` computes the same function with the same parameters, split into methods: `project` computes queries, keys and values, `rotate` adds positional information to queries and keys (nothing here), and `forward` attends with SDPA and merges the heads. The check below copies the weights of `nn.MultiheadAttention` and compares the outputs.

In [ ]:
class SelfAttention(nn.Module):
    # multi-head self-attention, the same function as nn.MultiheadAttention
    def __init__(self, dim: int, heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.heads, self.head_dim = heads, dim // heads
        self.qkv = nn.Linear(dim, 3 * dim)  # queries, keys and values
        self.out = nn.Linear(dim, dim)
        self.dropout = dropout

    def project(
        self, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # (B, L, dim) -> 3 x (B, heads, L, head_dim)
        B, L, _ = x.shape
        qkv = self.qkv(x).view(B, L, 3, self.heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        return q, k, v

    def rotate(
        self, q: torch.Tensor, k: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # positional information for queries and keys, none here
        return q, k

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        B, L, dim = x.shape
        q, k, v = self.project(x)
        q, k = self.rotate(q, k)
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=mask[:, None, None, :],  # (B, 1, 1, L): over heads and queries
            dropout_p=self.dropout if self.training else 0.0,
        )
        # (B, heads, L, head_dim) -> (B, L, dim)
        return self.out(out.transpose(1, 2).reshape(B, L, dim))


mha = Attention(256, heads=4).eval()
attention = SelfAttention(256, heads=4).eval()
attention.load_state_dict(
    {
        "qkv.weight": mha.mha.in_proj_weight,
        "qkv.bias": mha.mha.in_proj_bias,
        "out.weight": mha.mha.out_proj.weight,
        "out.bias": mha.mha.out_proj.bias,
    }
)
x, mask = torch.randn(2, 6, 256), torch.ones(2, 6, dtype=torch.bool)
mask[1, 4:] = False
assert torch.allclose(attention(x, mask)[mask], mha(x, mask)[mask], atol=1e-5)
print("the same outputs as nn.MultiheadAttention")

### Exercise 7: RoPE

We use the RoPE implementation from `transformers`, shared by LLaMA, Qwen, Mistral and many other models:
* `LlamaRotaryEmbedding` (created in `__init__`, no learned parameters) computes the cosines and sines of the angles: `self.rotary(q, positions)` takes positions with shape `(1, L)` and returns `cos` and `sin` with shape `(1, L, head_dim)` (`q` only sets their type and device)
* `apply_rotary_pos_emb(q, k, cos, sin)` rotates queries and keys `(B, heads, L, head_dim)` and returns them

Implement `RoPEAttention.rotate`: rotate queries and keys at positions `0, 1, ..., L - 1`.

In [ ]:
class RoPEAttention(SelfAttention):
    def __init__(self, dim: int, heads: int, dropout: float = 0.0) -> None:
        super().__init__(dim, heads, dropout)
        config = transformers.LlamaConfig(head_dim=self.head_dim, rope_theta=10000.0)
        self.rotary = LlamaRotaryEmbedding(config)

    def rotate(
        self, q: torch.Tensor, k: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # q, k: (B, heads, L, head_dim)
        ...

In [ ]:
attention = RoPEAttention(256, heads=4).eval()
check_block(attention, 256, num_params=count_params(SelfAttention(256, heads=4)))

# plain attention doesn't know positions: shuffling the tokens only shuffles the outputs
x, mask = torch.randn(1, 6, 256), torch.ones(1, 6, dtype=torch.bool)
order = torch.tensor([2, 0, 1, 5, 3, 4])
plain_attention = SelfAttention(256, heads=4).eval()
plain_attention.load_state_dict(attention.state_dict())
assert torch.allclose(
    plain_attention(x[:, order], mask), plain_attention(x, mask)[:, order], atol=1e-5
)
assert not torch.allclose(
    attention(x[:, order], mask), attention(x, mask)[:, order], atol=1e-3
), "with RoPE, the outputs should depend on the positions of tokens"

# padding at the end doesn't change the positions of real tokens
padded_x = torch.cat([x, torch.randn(1, 2, 256)], dim=1)
padded_mask = torch.cat([mask, torch.zeros(1, 2, dtype=torch.bool)], dim=1)
assert torch.allclose(
    attention(padded_x, padded_mask)[:, :6], attention(x, mask), atol=1e-5
), "padding should not change the outputs of real tokens"

print("OK")

RoPE removes the 128K parameters of the position table (512 x 256), but costs some time: queries and keys are rotated in every layer.

In [ ]:
def make_model() -> nn.Module:
    return Encoder(partial(RoPEAttention, heads=HEADS, dropout=DROPOUT), max_len=None)


model = train_step("2. RoPE", make_model)

## Step 3: Gated MLP (GeGLU)

The MLP holds 2/3 of the parameters of a Transformer block. [Shazeer (2020)](https://arxiv.org/abs/2002.05202) proposed **gated linear units (GLU)**: one projection, passed through the activation, is a "gate" that multiplies another one element-wise:

$$\mathrm{GeGLU}(x) = W_{down}\left(\mathrm{GELU}(W_{gate} x) \odot W_{up} x\right)$$

GeGLU uses GELU (ModernBERT, Gemma), SwiGLU uses SiLU (LLaMA, Qwen). Letting the network control, per token, how much of every feature passes works consistently better than a plain activation.

### Exercise 8: GeGLU

Implement `GeGLU` with `hidden = 8 * dim // 3`, which keeps the parameter count of the 4x MLP ($3 \cdot \frac{8}{3}d \cdot d = 8d^2$). Compute both $W_{gate} x$ and $W_{up} x$ with one linear layer `self.up` (`dim -> 2 * hidden`) and split its output with `chunk`. `self.down` maps `hidden -> dim`.

In [ ]:
class GeGLU(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()
        hidden = 8 * dim // 3
        self.up = ...
        self.down = ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        ...

In [ ]:
mlp = GeGLU(256)
assert mlp.up.out_features == 2 * 682 and mlp.down.in_features == 682
assert mlp(torch.randn(2, 5, 256)).shape == (2, 5, 256)
print(f"parameters: MLP {count_params(MLP(256))}, GeGLU {count_params(mlp)}")

x = torch.randn(4, 256)
with torch.no_grad():
    mlp.up.weight[682:] = 0  # zero "up" half: the output should not depend on x
    mlp.up.bias[682:] = 0
assert torch.allclose(mlp(x), mlp.down.bias.expand(4, -1)), (
    "the gate should multiply the second half of the projection"
)

print("OK")

In [ ]:
def make_model() -> nn.Module:
    return Encoder(
        partial(RoPEAttention, heads=HEADS, dropout=DROPOUT),
        make_mlp=GeGLU,
        max_len=None,
    )


model = train_step("3. GeGLU", make_model)

## Step 4: Mean pooling

BERT represents the sequence by the output embedding of `[CLS]`. This works well after pretraining, but trained from scratch, the model must learn to gather everything in one token. **Mean pooling**, the average output embedding of all real tokens, uses them all directly; most embedding models use it (Part 2).

### Exercise 9: Mean pooling

Implement `MeanPooling`: the average of `x` (`(B, L, dim)`) over the real tokens, using `mask` (`(B, L)`). Hint: convert the mask to the type of `x` and use broadcasting.

In [ ]:
class MeanPooling(nn.Module):
    def __init__(self, dim: int) -> None:
        super().__init__()

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        ...

In [ ]:
x = torch.randn(2, 6, 16)
mask = torch.tensor([[1, 1, 1, 0, 0, 0], [1, 1, 1, 1, 1, 1]], dtype=torch.bool)
pooled = MeanPooling(16)(x, mask)
assert pooled.shape == (2, 16)
assert torch.allclose(pooled[0], x[0, :3].mean(dim=0)), "average only the real tokens"
assert torch.allclose(pooled[1], x[1].mean(dim=0))

print("OK")

In [ ]:
def make_modern_model() -> nn.Module:
    return Encoder(
        partial(RoPEAttention, heads=HEADS, dropout=DROPOUT),
        make_mlp=GeGLU,
        make_pooling=MeanPooling,
        max_len=None,
    )


model = train_step("4. mean pooling", make_modern_model)

## Step 5: FlashAttention and unpadding

Our attention already uses SDPA. The plain implementation writes the full `(B, heads, L, L)` score matrix to GPU memory and reads it back, which is slow and memory-hungry for long sequences (Part 1, section 1.6). [FlashAttention](https://arxiv.org/abs/2205.14135) computes the same result in blocks in the fast on-chip memory, never storing the matrix. SDPA picks the best kernel ("backend") for the inputs, but its FlashAttention kernel doesn't support masks: with a padding mask, it falls back to a slower "memory-efficient" kernel.

Let's measure attention alone (forward + backward, 16 sequences of length L): the plain implementation (forced with `sdpa_kernel(SDPBackend.MATH)`), SDPA without a mask, and SDPA with a padding mask, as in our model.

In [ ]:
def benchmark_attention(fn: Callable[[], torch.Tensor], steps: int = 10) -> float:
    # average time of forward + backward pass, in milliseconds
    for _ in range(3):  # warmup
        fn().float().sum().backward()
    synchronize()
    start = time.time()
    for _ in range(steps):
        fn().float().sum().backward()
    synchronize()
    return (time.time() - start) / steps * 1000


print(f"{'L':>5s} {'plain':>9s} {'SDPA':>9s} {'SDPA + mask':>12s}")
for L in [128, 512, 1024, 2048] if device.type == "cuda" else [128, 256, 512]:
    shape = (16, 4, L, 64)  # (B, heads, L, head_dim)
    q, k, v = (
        torch.randn(shape, device=device, dtype=HALF_DTYPE).requires_grad_()
        for _ in range(3)
    )
    mask = torch.ones(16, 1, 1, L, dtype=torch.bool, device=device)
    with sdpa_kernel(SDPBackend.MATH):
        plain = benchmark_attention(
            lambda: F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        )
    fused = benchmark_attention(lambda: F.scaled_dot_product_attention(q, k, v))
    masked = benchmark_attention(
        lambda: F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    )
    print(f"{L:5d} {plain:7.2f}ms {fused:7.2f}ms {masked:10.2f}ms")

Fused kernels are several times faster for long sequences, but the padding mask takes away part of the gain. Padding wastes even more: in random batches, about 2/3 of all tokens are padding (see "Batches"), and the model computes embeddings, attention and MLPs for all of them.

**Unpadding** removes the padding tokens and processes a batch as one sequence of `T` real tokens, `(T, dim)` instead of `(B, L, dim)`. All layers except attention work on single tokens, so they work unchanged. Attention must not mix sequences: the "variable length" (varlen) FlashAttention takes the cumulative sequence lengths `cu_seqlens = [0, len_0, len_0 + len_1, ...]` and needs no mask. ModernBERT uses unpadding, and so do LLM inference engines like vLLM. Varlen FlashAttention needs an NVIDIA GPU since Ampere (e.g. RTX 30xx, A100); elsewhere (Colab T4, Mac, CPU), we pad back for attention only, so the other layers still gain.

`UnpaddedBatch` holds what attention needs after unpadding (see the comments). `UnpaddedAttention` gets the tokens `(T, dim)` and the `UnpaddedBatch` in place of the padding mask. It uses varlen FlashAttention if possible, otherwise SDPA on queries, keys and values padded back. Its `window` is used in step 6. Varlen attention has no attention dropout, which doesn't change the accuracy here.

In [ ]:
@dataclass
class UnpaddedBatch:
    mask: torch.Tensor  # (B, L), True for real tokens
    positions: torch.Tensor  # (T,), position of every token in its sequence
    cu_seqlens: torch.Tensor  # (B + 1,), int32, start of every sequence + total length
    max_len: int  # length of the longest sequence


# varlen FlashAttention: NVIDIA GPUs since Ampere (compute capability 8.0)
USE_VARLEN = (
    varlen_attn is not None
    and device.type == "cuda"
    and torch.cuda.get_device_capability()[0] >= 8
)
print(
    f"varlen FlashAttention: {'yes' if USE_VARLEN else 'no, padding for attention only'}"
)


class UnpaddedAttention(RoPEAttention):
    window: int | None = None  # sliding window size, None = global attention (step 6)

    def forward(self, x: torch.Tensor, batch: UnpaddedBatch) -> torch.Tensor:
        # x: (T, dim) tokens of all sequences in the batch, without padding
        T, dim = x.shape
        q, k, v = self.qkv(x).view(T, 3, self.heads, self.head_dim).unbind(dim=1)
        # RoPE with the position of every token within its sequence
        cos, sin = self.rotary(q, batch.positions[None])  # (1, T, head_dim)
        q, k = apply_rotary_pos_emb(q, k, cos[0], sin[0])  # (T, heads, head_dim)
        if USE_VARLEN and x.is_cuda:
            # half the window on each side, -1 = no limit
            w = (self.window // 2, self.window // 2) if self.window else (-1, -1)
            out = varlen_attn(
                q, k, v, batch.cu_seqlens, batch.cu_seqlens, batch.max_len, batch.max_len,
                window_size=w,
            )  # fmt: skip
        else:
            # pad back to (B, heads, L, head_dim) for attention only
            q, k, v = (repad(t, batch.mask).transpose(1, 2) for t in (q, k, v))
            mask = batch.mask[:, None, None, :]
            if self.window:
                mask = mask & sliding_window_mask(mask.shape[-1], self.window, x.device)
            p = self.dropout if self.training else 0.0
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p)
            out = out.transpose(1, 2)[batch.mask]
        return self.out(out.reshape(T, dim))

`unpad` converts padded token ids `(B, L)` to the real tokens `(T,)` and the `UnpaddedBatch`: boolean indexing `ids[mask]` takes the real tokens row by row, and the positions are taken the same way. `repad` is the inverse, `(T, ...)` to `(B, L, ...)` with zeros at padding.

In [ ]:
def unpad(ids: torch.Tensor) -> tuple[torch.Tensor, UnpaddedBatch]:
    mask = ids != PAD
    lengths = mask.sum(dim=1)
    positions = torch.arange(ids.shape[1], device=ids.device).expand_as(ids)[mask]
    cu_seqlens = F.pad(
        lengths.cumsum(dim=0), (1, 0)
    ).int()  # [0, len_0, len_0 + len_1, ...]
    return ids[mask], UnpaddedBatch(mask, positions, cu_seqlens, int(lengths.max()))


def repad(x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    out = x.new_zeros(*mask.shape, *x.shape[1:])
    out[mask] = x
    return out


ids = torch.tensor([[CLS, 7, 8, 9, SEP], [CLS, 5, SEP, PAD, PAD]])
tokens, batch = unpad(ids)
print("tokens:", tokens.tolist())
print("positions:", batch.positions.tolist())
print("cu_seqlens:", batch.cu_seqlens.tolist())

### Exercise 10: Unpadded encoder

Implement `UnpaddedEncoder.forward`: unpad the token ids, embed them (no position embeddings, RoPE uses `batch.positions`), dropout, blocks (with the `UnpaddedBatch` in place of the mask), final norm, then pad back with `repad` for pooling and the head.

In [ ]:
class UnpaddedEncoder(Encoder):
    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        ...

In [ ]:
def make_unpadded_model() -> nn.Module:
    return UnpaddedEncoder(
        partial(UnpaddedAttention, heads=HEADS, dropout=DROPOUT),
        make_mlp=GeGLU,
        make_pooling=MeanPooling,
        max_len=None,
    )


# the unpadded model computes the same function as the padded one
torch.manual_seed(0)
padded_model = make_modern_model().eval()
unpadded_model = make_unpadded_model().eval()
unpadded_model.load_state_dict(padded_model.state_dict())
x, _ = next(make_batches(rna_ids["validation"], rna_labels["validation"], 16))
with torch.no_grad():
    assert torch.allclose(unpadded_model(x), padded_model(x), atol=1e-4), (
        "the unpadded model should give the same outputs as the padded one"
    )

print("OK")

In [ ]:
model = train_step("5. unpadding", make_unpadded_model)

Unpadding keeps the accuracy of step 4 and about halves the training time; without varlen FlashAttention (T4, Mac), the gain is smaller. LLM training goes further with **packing**: sequences are concatenated into rows of fixed length and separated by varlen attention, so no compute is wasted.

## Step 6: Sliding window + global attention

Most information is local, so ModernBERT uses **sliding window attention**: every token attends only to the `window` nearest tokens (`window / 2` on each side), so the cost grows linearly with length. Every third layer stays **global**, so that information can flow between distant tokens. Gemma, Mistral and GPT-OSS mix local and global layers in the same way. The speedup needs a kernel that skips the blocks outside the window, like FlashAttention with `window_size`.

nRC sequences are short (median 41 tokens), so a window of 64 covers most of them, and we expect no change in accuracy or speed. The benchmark after training shows the effect for long sequences.

### Exercise 11: Sliding window encoder

Implement:
* `sliding_window_mask(length, window, device)`: a boolean `(length, length)` mask, `True` where query $i$ may attend to key $j$, i.e. $|i - j| \le$ `window // 2`. It is used by `UnpaddedAttention` without varlen FlashAttention
* `SlidingWindowEncoder`: after creating the model with `super().__init__`, set `window` of the attention layers: global attention (`None`) in every `global_every`-th layer starting from the first one (layers 0, 3, ... for `global_every=3`), and `window` in all others

In [ ]:
def sliding_window_mask(
    length: int, window: int, device: torch.device | None = None
) -> torch.Tensor:
    ...


class SlidingWindowEncoder(UnpaddedEncoder):
    def __init__(
        self, *args, window: int = 64, global_every: int = 3, **kwargs
    ) -> None:
        super().__init__(*args, **kwargs)
        ...

In [ ]:
mask = sliding_window_mask(6, 4)
assert mask.shape == (6, 6) and mask.dtype == torch.bool
assert mask[0].tolist() == [True, True, True, False, False, False], (
    "2 tokens on each side"
)
assert torch.equal(mask, mask.T)

model = SlidingWindowEncoder(
    partial(UnpaddedAttention, heads=HEADS), window=8, global_every=3
)
windows = [block.attention.window for block in model.blocks]
assert windows == [None, 8, 8, None, 8, 8], f"got {windows}"

# local attention: a token far away (beyond all windows) has no influence on the output
model.eval()
x = torch.randn(1, 30, 256)
_, batch = unpad(torch.ones(1, 30, dtype=torch.long))
attention = model.blocks[1].attention
x_changed = x.clone()
x_changed[0, 20] += 1
out, out_changed = attention(x[0], batch), attention(x_changed[0], batch)
changed = (out - out_changed).abs().amax(dim=-1) > 1e-5
assert changed[16:25].all() and not changed[:16].any() and not changed[25:].any(), (
    "token 20 should influence only tokens 16-24 with window 8"
)

print("OK")

In [ ]:
def make_model() -> nn.Module:
    return SlidingWindowEncoder(
        partial(UnpaddedAttention, heads=HEADS, dropout=DROPOUT),
        make_mlp=GeGLU,
        make_pooling=MeanPooling,
        max_len=None,
        window=64,
    )


model = train_step("6. sliding window", make_model)

Attention speed for long sequences: global attention vs a window of 128 tokens (as in ModernBERT), for batches of about 16K tokens.

In [ ]:
print(f"{'L':>5s} {'global':>9s} {'window 128':>11s}")
for L in [512, 1024, 2048, 4096, 8192] if device.type == "cuda" else [256, 512, 1024]:
    B = max(1, 16384 // L) if device.type == "cuda" else 4
    attention = UnpaddedAttention(256, heads=4).to(device)
    x = torch.randn(B * L, 256, device=device, requires_grad=True)
    _, batch = unpad(torch.ones(B, L, dtype=torch.long, device=device))
    times = []
    for window in (None, 128):
        attention.window = window
        with autocast():
            times.append(benchmark_attention(lambda: attention(x, batch)))
    print(f"{L:5d} {times[0]:7.2f}ms {times[1]:9.2f}ms")

With a constant number of tokens per batch, the time of global attention grows linearly with L, while with varlen FlashAttention, the sliding window time stays constant (both include the same QKV and output projections). Masking the scores outside the window gives the same result but computes the whole matrix, so without varlen FlashAttention the window is not faster at all.